[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/06_00_main_intuition.ipynb)

# Vision intuition: three small demos

**Notebook:** `06_00_main_intuition`

Three ideas, no neural networks yet (we'll get there!):

1. **An image is just a number array.** Everything else in this module is operations *on that array.*
2. **A convolution is a tiny matrix slid across the image.** That's it. We code one by hand.
3. **Pooling shrinks the picture but keeps the structure.** That's why CNNs can stack layer after layer without exploding in size.

Run the cells, look at the pictures, build the mental model. The rest of Module 6 builds on these three ideas.

## Setup

Just numpy, matplotlib, and one sample image that ships with scikit-learn. No downloads, no GPU... though they make everything better.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_sample_image

## Part 1: An image is a number array

Load a picture and look at what it actually *is* in memory. `load_sample_image` ships a couple of stock photos with scikit-learn. We use the flower one because it has obvious edges and color contrast, both useful in Part 2.

What to notice:
- `shape` is `(height, width, 3)`, three channels for red/green/blue
- `dtype` is `uint8`, integers from 0 (black) to 255 (full intensity)
- the image *is* the array. There is no extra structure.

In [ ]:
img = load_sample_image("flower.jpg")

print("shape:", img.shape)
print("dtype:", img.dtype)
print("min / max pixel value:", img.min(), "/", img.max())

plt.figure(figsize=(6, 4))
plt.imshow(img)
plt.axis("off")
plt.title(f"raw image, {img.shape[1]}x{img.shape[0]} pixels, 3 channels")
plt.show()

### What's actually in the array?

Print a tiny patch from the middle of the image. Each cell is one pixel; each pixel has three numbers (R, G, B). That's the whole story.

In [ ]:
h, w, _ = img.shape
patch = img[h // 2 : h // 2 + 4, w // 2 : w // 2 + 4]   # 4x4 pixel patch
print("patch shape:", patch.shape, "\n")
print(patch)

### Manipulating pixels is just numpy

Three classic 'image processing' operations, all one-liners on the array:
- **Channel split.** Show only the red channel.
- **Grayscale.** Average the three channels.
- **Invert.** Subtract from 255.

No special library. The 'image' is just numbers.

In [ ]:
red_only = img.copy()
red_only[..., 1] = 0   # zero out green
red_only[..., 2] = 0   # zero out blue

gray = img.mean(axis=2).astype(np.uint8)

inverted = 255 - img

fig, axs = plt.subplots(1, 4, figsize=(14, 4))
for ax, im, title in zip(
    axs,
    [img, red_only, gray, inverted],
    ["original", "red channel only", "grayscale (mean of RGB)", "inverted (255 - x)"],
):
    ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Part 2: Convolution is a tiny matrix slid across the image

A *convolution* (well, technically a cross-correlation, but everyone calls it convolution... so why push against the wind) is one operation:

1. Pick a small matrix, the **kernel** or **filter** (3x3, 5x5).
2. Slide it across the image. At each position, multiply element-wise with the patch underneath, sum, write the result to the output.

That's it. Different kernels detect different things. Edges, blur, sharpening, all come from changing the nine numbers in a 3x3 matrix.

We'll write the loop ourselves first, then look at common kernels.

In [ ]:
def convolve2d(image, kernel):
    """Apply a 2D kernel to a grayscale image. Pure-Python loop, intentionally slow and readable."""
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2                                  # padding so output is the same size
    padded = np.pad(image, ((ph, ph), (pw, pw)), mode="edge")
    out = np.zeros_like(image, dtype=np.float32)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            patch = padded[i : i + kh, j : j + kw]
            out[i, j] = (patch * kernel).sum()                  # the whole operation is on this line
    return out

### A first kernel: vertical edge detector (Sobel)

Pixels on the left side of the kernel get multiplied by negative weights, pixels on the right by positive weights. Where the image has a left-to-right brightness jump (a vertical edge), the patch sums to a big number. Where the image is flat (no edge), things cancel and the sum is near zero.

Gray = no edge, bright/dark = strong edge.

In [ ]:
sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype=np.float32)

edges_x = convolve2d(gray.astype(np.float32), sobel_x)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].imshow(gray, cmap="gray");          axs[0].set_title("input (grayscale)");                axs[0].axis("off")
axs[1].imshow(edges_x, cmap="gray");       axs[1].set_title("after sobel_x: vertical edges");    axs[1].axis("off")
plt.tight_layout()
plt.show()

### A small zoo of kernels

Same image, four different kernels. Each one extracts a different *property*: horizontal edges, smoothness, sharpness, or just the original.

**The CNN insight (preview).** Hand-designing useful kernels is an art (people did this for decades, see SIFT, HOG, the Sobel/Prewitt/Scharr family). The thing a CNN does is **learn the kernel weights from data**, instead of designing them. Same operation, just with the nine numbers chosen by gradient descent rather than by a human.

In [ ]:
kernels = {
    "identity (does nothing)": np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=np.float32),
    "sobel_y (horizontal edges)": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32),
    "box blur": np.ones((3, 3), dtype=np.float32) / 9.0,
    "sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32),
}

fig, axs = plt.subplots(1, len(kernels), figsize=(4 * len(kernels), 4))
for ax, (name, k) in zip(axs, kernels.items()):
    out = convolve2d(gray.astype(np.float32), k)
    ax.imshow(out, cmap="gray")
    ax.set_title(name, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Part 3: Pooling shrinks the picture but keeps the structure

Convolutions don't change image size by much (we used `mode='edge'` padding to keep it identical). To stack many layers without blowing up memory and parameters, CNNs interleave **pooling**, a downsampling step that summarizes each small region into a single number.

The most common one is **2x2 max pooling**: take each 2x2 block of pixels, keep the max, throw away the other three. Output is half the height and half the width.

Counterintuitive but useful: even with 75% of the pixels gone, the picture is still recognizable. A CNN doesn't need every pixel, it needs the *strongest signal in each region*.

In [ ]:
def max_pool_2x2(image):
    h, w = image.shape
    h2, w2 = h - (h % 2), w - (w % 2)                          # crop to even dims for clean reshape
    cropped = image[:h2, :w2]
    return cropped.reshape(h2 // 2, 2, w2 // 2, 2).max(axis=(1, 3))

p1 = max_pool_2x2(gray)
p2 = max_pool_2x2(p1)
p3 = max_pool_2x2(p2)

fig, axs = plt.subplots(1, 4, figsize=(14, 4))
for ax, im, label in zip(
    axs,
    [gray, p1, p2, p3],
    [f"original {gray.shape}", f"1x pool {p1.shape}", f"2x pool {p2.shape}", f"3x pool {p3.shape}"],
):
    ax.imshow(im, cmap="gray")
    ax.set_title(label, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Takeaways

1. **An image is a number array.** `(H, W, 3)` of `uint8`. Anything you can do to a numpy array, you can do to an image.
2. **A convolution is one short loop.** Slide a small matrix, multiply, sum. Different kernels extract different properties.
3. **Pooling discards 75% of the pixels and the picture survives.** That's why CNNs can stack layers cheaply.

These three things, an array, a sliding kernel, and a downsampling step, are *the entire architectural vocabulary* of a CNN. The next notebook ([`06_01_main_classical.ipynb`](06_01_main_classical.ipynb)) shows what people built with hand-designed features before deep learning. The one after ([`06_02_main_cnn.ipynb`](06_02_main_cnn.ipynb)) lets the network learn its own filters from data.